## 🎯 Learning Objectives
* Design and implement a LangChain agent capable of performing web searches.
* Integrate conversational memory into a LangChain agent to maintain context across turns.
* Evaluate the agent's ability to answer complex research questions and follow-up queries.


## Exercise: Build a Research Agent with Web Search and Memory

### Task Description

Your goal is to construct a sophisticated LangChain agent that acts as a "Research Assistant." This agent should be capable of answering complex questions by leveraging web search capabilities and maintaining conversational context through memory. The agent will simulate a research session where a user asks initial questions and then follow-up questions based on the agent's previous responses.

### Requirements

1.  **Agent Architecture**: Utilize LangChain's latest LCEL-based agent architecture (`create_react_agent`).
2.  **Web Search Tool**: Integrate a robust web search tool (e.g., Tavily Search API) to allow the agent to find up-to-date information from the internet.
3.  **Conversational Memory**: Implement conversational memory to enable the agent to remember previous turns in the conversation. This is crucial for answering follow-up questions without needing to re-state the full context.
4.  **LLM Integration**: Use a modern, capable Large Language Model (LLM) for the agent's reasoning and response generation (e.g., OpenAI's GPT-4o, Anthropic's Claude 3.5 Sonnet, or Google's Gemini 1.5 Pro).
5.  **Prompt Engineering**: Design an effective prompt that guides the agent to act as a helpful research assistant, encouraging it to use its tools when necessary and provide concise, accurate answers.
6.  **Error Handling (Optional but Recommended)**: Consider basic error handling for tool calls or LLM responses, though not strictly required for this exercise.

### Evaluation Criteria

*   **Correctness**: Does the agent correctly implement web search and memory?
*   **Accuracy**: Does the agent provide accurate answers to research questions, especially those requiring web search?
*   **Contextual Understanding**: Can the agent effectively answer follow-up questions based on previous turns, demonstrating proper memory integration?
*   **Code Quality**: Is the code clean, well-commented, and easy to understand?
*   **Robustness**: Does the agent handle various types of questions, including those that explicitly require web search and those that can be answered from memory?

### Example Interaction Flow

```
User: "What is the capital of France?"
Agent: "The capital of France is Paris."

User: "What is its population?"
Agent: "As of 2024, the population of Paris is approximately 2.1 million people within the city proper, and over 12 million in the greater metropolitan area."

User: "Tell me about the latest advancements in AI in 2026."
Agent: (Uses web search) "In 2026, key advancements in AI include breakthroughs in multimodal foundation models, significant improvements in autonomous agent capabilities, and widespread adoption of explainable AI techniques in enterprise solutions..."
```


In [ ]:
# Install necessary packages (ensure you have these installed in your environment)
# !pip install -qU langchain langchain-openai tavily-python

import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import Tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import create_react_agent, AgentExecutor
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage

# --- Configuration --- 
# Set your API keys as environment variables
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
# os.environ["TAVILY_API_KEY"] = "YOUR_TAVILY_API_KEY"

# Check if API keys are set
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY environment variable not set.")
if not os.getenv("TAVILY_API_KEY"):
    raise ValueError("TAVILY_API_KEY environment variable not set.")

print("Setup complete. API keys are loaded from environment variables.")
print("You are ready to build your research agent!")

# Initialize the LLM (using GPT-4o as a modern choice for 2026)
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Initialize the web search tool
# Tavily is a good choice for general web search, offering good results and ease of use.
web_search_tool = TavilySearchResults(max_results=5)

# Define the tools available to the agent
tools = [
    web_search_tool,
]

# This dictionary will store chat histories for different sessions.
# In a real application, this would be a persistent database.
store = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

print("LLM and Web Search Tool initialized. Session history store created.")


### Your Turn: Implement the Research Agent

Now it's your turn to assemble the research agent using the initialized components. Follow these steps:

1.  **Create the Agent Prompt**: Define a `ChatPromptTemplate` that includes:
    *   A system message instructing the agent on its role (Research Assistant).
    *   A `MessagesPlaceholder` for `chat_history` to inject past conversation turns.
    *   A `HumanMessage` template for the current input.
    *   A `MessagesPlaceholder` for `agent_scratchpad` to allow the agent to show its thought process and tool usage.

2.  **Construct the Agent**: Use `create_react_agent` to combine your `llm`, `tools`, and the `prompt` you just created.

3.  **Create the Agent Executor**: Instantiate `AgentExecutor` with your agent and tools. Set `verbose=True` to see the agent's thought process.

4.  **Integrate Memory**: Wrap your `AgentExecutor` with `RunnableWithMessageHistory` using the `get_session_history` function provided in the setup cell. This will enable the agent to remember past interactions.

5.  **Test Your Agent**: Run your agent with a few questions, including:
    *   A question requiring web search (e.g., "What are the latest breakthroughs in quantum computing as of 2026?").
    *   A follow-up question that relies on the previous turn's context (e.g., "Who are the key researchers in that field?").
    *   A simple factual question that might not need web search.

Good luck!


In [ ]:
# --- Reference Solution --- 

# 1. Create the Agent Prompt
# The prompt is crucial for guiding the agent's behavior.
# - System message sets the persona.
# - chat_history placeholder allows memory to be injected.
# - agent_scratchpad placeholder is where the agent's thoughts and tool outputs go.
agent_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful AI Research Assistant. Your primary goal is to provide accurate and up-to-date information by leveraging web search when necessary. "
            "Always strive to answer questions thoroughly and concisely. If a question requires current information, use your web search tool. "
            "Maintain context from previous turns to answer follow-up questions effectively."
        ),
        MessagesPlaceholder("chat_history"), # This is where the memory will be injected
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad"), # This is where the agent's thoughts and tool outputs will be displayed
    ]
)

# 2. Construct the Agent
# create_react_agent is a convenient way to set up a ReAct agent with LCEL.
# It takes the LLM, the tools it can use, and the prompt.
agent = create_react_agent(llm, tools, agent_prompt)

# 3. Create the Agent Executor
# The AgentExecutor is responsible for running the agent, handling tool calls, and managing the loop.
# verbose=True helps in debugging and understanding the agent's thought process.
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 4. Integrate Memory with RunnableWithMessageHistory
# This runnable wraps the agent_executor and automatically manages chat history.
# get_session_history is a function that retrieves or creates a ChatMessageHistory for a given session ID.
# input_messages_key specifies which key in the input dictionary contains the current user message.
# history_messages_key specifies which key in the prompt expects the chat history.
agent_with_memory = RunnableWithMessageHistory(
    agent_executor,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

print("\n--- Agent with Memory Initialized. Starting Test Interactions ---\n")

# 5. Test Your Agent
# We'll use a fixed session ID for demonstration. In a real app, this would be dynamic.
session_id = "test_session_123"

# Test 1: Question requiring web search
print("\n--- Test 1: Initial Web Search Question ---")
response1 = agent_with_memory.invoke(
    {"input": "What are the latest breakthroughs in quantum computing as of 2026?"},
    config={"configurable": {"session_id": session_id}}
)
print(f"\nAgent: {response1['output']}")

# Test 2: Follow-up question relying on memory and potentially another web search
print("\n--- Test 2: Follow-up Question (Memory + Web Search) ---")
response2 = agent_with_memory.invoke(
    {"input": "Who are the key researchers or companies leading these advancements?"},
    config={"configurable": {"session_id": session_id}}
)
print(f"\nAgent: {response2['output']}")

# Test 3: A simple factual question that might not need web search, but still uses memory
print("\n--- Test 3: Simple Factual Question (Memory) ---")
response3 = agent_with_memory.invoke(
    {"input": "What is the primary challenge in scaling quantum computers?"},
    config={"configurable": {"session_id": session_id}}
)
print(f"\nAgent: {response3['output']}")

# Test 4: A new session to demonstrate memory isolation
print("\n--- Test 4: New Session (Memory Isolation) ---")
new_session_id = "new_test_session_456"
response4 = agent_with_memory.invoke(
    {"input": "What was the first question I asked in this new session?"},
    config={"configurable": {"session_id": new_session_id}}
)
print(f"\nAgent: {response4['output']}")

print("\n--- Agent testing complete ---")
